# Day 5 — Data Consistency & Validation

## Objectives
- Check categorical consistency
- Validate numerical ranges
- Check binary Yes/No fields
- Verify data types
- Detect invalid or unexpected values
- Perform final dataset validation
- Prepare the dataset for final cleaning

In [22]:
import pandas as pd
import numpy as np


In [17]:
# load datset
df = pd.read_csv('/content/healthcare_patients.csv')
df.shape
df.head()


,patient_id,age,gender,blood_pressure,heart_rate,bmi,smoking_status,diabetes,hypertension,disease,admission_type,length_of_stay,medication_count,treatment_cost,readmission,risk_score
0,103475,84,Female,144.0,81.0,21.2,Current,No,No,Healthy/Preventive,Urgent,5,2.0,19148.0,No,45.9
1,105328,42,Male,130.0,104.0,34.2,Never,No,No,Hypertension,Emergency,7,3.0,28092.0,No,42.5
2,102009,56,Female,127.0,71.0,28.8,Current,Yes,Yes,Other,Urgent,7,8.0,20706.0,No,56.4
3,104186,54,Female,148.0,70.0,31.8,Never,No,Yes,Diabetes,Emergency,5,6.0,25249.0,No,62.9
4,108876,57,Male,130.0,96.0,26.7,Never,No,No,Healthy/Preventive,Elective,3,4.0,14612.0,No,28.2


In [18]:
# check missing values
df.isna().sum()
df.isnull().sum().sum()

np.int64(750)

In [19]:
# Check categorical consistency
df["gender"].value_counts()
df["smoking_status"].value_counts(dropna=False)
df["disease"].value_counts()
df["admission_type"].value_counts()

,count
admission_type,
Elective,4375
Emergency,3171
Urgent,2474


In [20]:
# Check Yes/No fields
df["diabetes"].value_counts()
df["hypertension"].value_counts()
df["readmission"].value_counts()

,count
readmission,
No,7952
Yes,2068


In [21]:
# Check for unexpected categorical values
valid_gender = ["Male", "Female", "Other"]

invalid_gender = df[
    ~df["gender"].isin(valid_gender)
]

invalid_gender.shape

(0, 16)

In [24]:
valid_smoking = ["Never", "Former", "Current"]

invalid_smoking = df[
    df["smoking_status"].notna() &
    ~df["smoking_status"].isin(valid_smoking)
]

invalid_smoking.shape

(0, 16)

In [25]:
valid_disease = [
    "Hypertension",
    "Healthy/Preventive",
    "Diabetes",
    "Heart Disease",
    "Other",
    "Respiratory Disease",
    "Kidney Disease",
    "Cancer"
]

invalid_disease = df[
    ~df["disease"].isin(valid_disease)
]

invalid_disease.shape

(0, 16)

In [34]:
# Validate numerical ranges
# age
df["age"].min(), df["age"].max()
invalid_age = df[
    (df["age"] < 18) |
    (df["age"] > 85)
]

print(invalid_age.shape)

# blood pressure
invalid_bp = df[
    df["blood_pressure"].notna() &
    (
        (df["blood_pressure"] < 90) |
        (df["blood_pressure"] > 190)
    )
]

print(invalid_bp.shape)

# heart rate
invalid_hr = df[
    df["heart_rate"].notna() &
    (
        (df["heart_rate"] < 45) |
        (df["heart_rate"] > 127)
    )
]

print(invalid_hr.shape)

# BMI
invalid_bmi = df[
    df["bmi"].notna() &
    (
        (df["bmi"] < 15) |
        (df["bmi"] > 45)
    )
]

print(invalid_bmi.shape)

# length of stay
invalid_los = df[
    (df["length_of_stay"] < 1) |
    (df["length_of_stay"] > 22)
]

print(invalid_los.shape)

# medication count
invalid_medication = df[
    df["medication_count"].notna() &
    (
        (df["medication_count"] < 0) |
        (df["medication_count"] > 12)
    )
]

print(invalid_medication.shape)

# treatment cost
invalid_cost = df[
    df["treatment_cost"].notna() &
    (df["treatment_cost"] < 0)
]

print(invalid_cost.shape)

# risk score
invalid_risk = df[
    (df["risk_score"] < 1) |
    (df["risk_score"] > 100)
]

print(invalid_risk.shape)

(0, 16)
(0, 16)
(0, 16)
(0, 16)
(0, 16)
(0, 16)
(0, 16)
(0, 16)


In [38]:
# check datatypes
df.dtypes

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10020 entries, 0 to 10019
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   patient_id        10020 non-null  int64  
 1   age               10020 non-null  int64  
 2   gender            10020 non-null  object 
 3   blood_pressure    9870 non-null   float64
 4   heart_rate        9900 non-null   float64
 5   bmi               9840 non-null   float64
 6   smoking_status    9920 non-null   object 
 7   diabetes          10020 non-null  object 
 8   hypertension      10020 non-null  object 
 9   disease           10020 non-null  object 
 10  admission_type    10020 non-null  object 
 11  length_of_stay    10020 non-null  int64  
 12  medication_count  9900 non-null   float64
 13  treatment_cost    9940 non-null   float64
 14  readmission       10020 non-null  object 
 15  risk_score        10020 non-null  float64
dtypes: float64(6), int64(3), object(7)
memor

In [39]:
# Check duplicate patient IDs
df["patient_id"].duplicated().sum()

np.int64(20)

In [42]:
# Now create the final cleaned dataset
df_cleaned = df.copy()

# hendeling missing numerical values
numerical_missing_columns = [
    "blood_pressure",
    "heart_rate",
    "bmi",
    "medication_count",
    "treatment_cost"
]

for column in numerical_missing_columns:
    df_cleaned[column] = df_cleaned[column].fillna(
        df_cleaned[column].median()
    )

# Handle missing smoking status
df_cleaned["smoking_status"] = df_cleaned["smoking_status"].fillna(df_cleaned["smoking_status"].mode()[0])

# remove duplicate record
df_cleaned = df_cleaned.drop_duplicates()

In [43]:
# final validation
print("Shape:", df_cleaned.shape)

print(
    "Missing values:",
    df_cleaned.isnull().sum().sum()
)

print(
    "Duplicate rows:",
    df_cleaned.duplicated().sum()
)

print(
    "Duplicate patient IDs:",
    df_cleaned["patient_id"].duplicated().sum()
)

Shape: (10000, 16)
Missing values: 0
Duplicate rows: 0
Duplicate patient IDs: 0


In [44]:
# save the final clean dataset
import os

os.makedirs("/content/data/processed", exist_ok=True)

df_cleaned.to_csv(
    "/content/data/processed/cleaned_healthcare_patients.csv",
    index=False
)

In [45]:
# verify cleaned dataset
pd.read_csv(
    "/content/data/processed/cleaned_healthcare_patients.csv"
).shape

(10000, 16)

## Day 5 Observations

- Categorical variables were checked for unexpected values.
- Gender contained only valid categories: Male, Female, and Other.
- Smoking status contained valid categories: Never, Former, and Current.
- Disease categories were consistent with the expected values.
- Admission types were consistent with the expected categories.
- Diabetes, hypertension, and readmission contained valid Yes/No values.
- Numerical variables were checked against their expected ranges.
- No invalid numerical values were detected.
- Missing numerical values were handled using median imputation.
- Missing smoking status values were handled using the mode.
- 20 duplicate records were removed.
- Final dataset contains 10,000 rows and 16 columns.
- Final validation confirmed 0 missing values and 0 duplicate records.
- The final cleaned dataset was saved for further exploratory data analysis.